# Set up a rectangular regional CESM-MOM6 run

Generating new case from partially generated one, not to re-generate same grids etc.

### Set up paths

In [ ]:
import os
from pathlib import Path
# CESM case (experiment) name
casename = "oleanderNWA12_2006-2026"

# CESM source root (you should not need to change this path)
cesmroot = Path(os.path.expandvars("/glade/work/emilanese/BaskNWA/model/CESM"))

# Place where all your input files go 
inputdir = Path("/glade/work/emilanese/croc_input") / casename
    
# CESM case directory
caseroot = Path("/glade/work/emilanese/croc_cases") / casename

### Loading previously generated grids

In [ ]:
import xarray as xr
from CrocoDash.grid import Grid
from CrocoDash.topo import Topo
from CrocoDash.vgrid import VGrid

# Paths
hgrid_path = Path("/glade/work/emilanese/NWA_Fred_grids/ocean_hgrid.nc")
bathymetry_path = Path("/glade/work/emilanese/NWA_Fred_grids/ocean_topo.nc")
vgrid_path = "/glade/work/emilanese/NWA_Fred_grids/vgrid_75_2m.nc"

# Horizontal grid
grid = Grid.from_supergrid(hgrid_path)

# Topography
with xr.open_dataset(bathymetry_path) as ds:
    min_depth = ds.depth.min().item() #ds.attrs["min_depth"]
topo = Topo.from_topo_file(
    grid = grid,
    topo_file_path=bathymetry_path,
    min_depth = min_depth,
)

# Vertical grid
vgrid  = VGrid.from_file(vgrid_path)

#### Verify topography

In [ ]:
topo.depth.plot()

#### Verify vertical grid

In [ ]:
import matplotlib.pyplot as plt
# Create the plot
for depth in vgrid.zi:
    plt.axhline(y=depth, linestyle='-')  # Horizontal lines

plt.ylim(max(vgrid.zi) + 10, min(vgrid.zi) - 10)  # Invert y-axis so deeper values go down
plt.ylabel("Depth")
plt.title("Vertical Grid")
plt.show()

### Erase disconnected basins from topography

NB: If you did this step already, they shold already be disconnected.

In [ ]:
%matplotlib ipympl
from CrocoDash.topo_editor import TopoEditor
TopoEditor(topo)

## Create the case

Using name and paths generated at beginning of this notebook

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = 'P93300012',
    override = True,
    machine = "derecho",
    compset = "CR_JRA" # This is the alias of the compset, the longname (which is printed when you run this command) is 1850_DATM%JRA_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_SWAV. Feel free to use either way!
)

# Section 3: Prepare ocean forcing data

We need to cut out our ocean forcing. The package expects an initial condition and one time-dependent segment per non-land boundary. Naming convention is `"east_unprocessed"` for segments and `"ic_unprocessed"` for the initial condition.

In this notebook, we are forcing with the Copernicus Marine "Glorys" reanalysis dataset. There's a function in the `CrocoDash` package, called `configure_forcings`, that generates a bash script to download the correct boundary forcing files for your experiment. First, you will need to create an account with Copernicus, and then call `copernicusmarine login` to set up your login details on your machine. Then you can run the `get_glorys_data.sh` bash script.

## Step 3.1 Configure Initial Conditions and Forcings


In [ ]:
case.configure_forcings(
    date_range = ["2006-01-01 00:00:00", "2026-01-01 00:00:00"],
    boundaries=["north","south","east","west"],
    function_name="get_glorys_data_script_for_cli"
)

##  Step 3.3: Process forcing data

In this final step, we call the `process_forcings` method of CrocoDash to cut out and interpolate the initial condition as well as all boundaries. CrocoDash also updates MOM6 runtime parameters and CESM xml variables accordingly.

In [ ]:
import json
 
config_path = case.extract_forcings_path / "config.json"
with open(config_path) as f:
    config = json.load(f)
    config["basic"]["general"]["step"] = 30  # 30 days per chunk
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

In [ ]:
case.process_forcings()

In [ ]:
print("You can now build and run your case at",caseroot)

# Section 4: Build and run the case

After completing the previous steps, you are ready to build and run your CESM case. Begin by navigating to the case root directory specified during the case creation. Before proceeding, review the `user_nl_mom` file located in the case directory. This file contains MOM6 parameter settings that were automatically generated by CrocoDash. Carefully examine these parameters and make any necessary adjustments to fine-tune the model for your specific requirements. While CrocoDash aims to provide a solid starting point, further tuning and adjustments are typically necessary to improve the model for your use case.

Once you have reviewed and modified the parameters as needed, you can build and execute the case using the following commands: 
```
qcmd -- ./case.build
./case.submit
```